In [94]:
"""
Telco Customer Churn Dataset: Machine Learning Analysis
Models used: Decision Tree and SVM (Support Vector Machine)

Dataset source: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
"""

'\nTelco Customer Churn Dataset: Machine Learning Analysis\nModels used: Decision Tree and SVM (Support Vector Machine)\n\nDataset source: https://www.kaggle.com/datasets/blastchar/telco-customer-churn\n'

In [95]:
# importing python libraries
import pandas as pd
import numpy as np

In [96]:
# importing scikit-learn for ML
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

## Step 1: Loading the dataset

In [97]:
# loading the dataset
df = pd.read_csv('telco.csv')

## Step 2: Analysis and preprocessing of the data

In [98]:
# short descriptive analysis of the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [99]:
# checking for any missing values
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [100]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [101]:
# dropping the customer id column as it carries no predictive value
df = df.drop(columns=['customerID'])

In [102]:
# confirming that the customer id column was dropped successfully
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [103]:
# TotalCharges column stored as text because 11 rows have blank strings (these are customers with tenure = 0, brand new customers who
# haven't been billed, filling with 0 is the fix)
# non numeric or blank values will be converted to NaN because of "errors='coerce'"
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [104]:
# confirming the changes
df['TotalCharges'].isnull().sum()
df['TotalCharges'].dtype

dtype('float64')

In [105]:
# encoding the target column 'Churn' from 'Yes/No' to 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [106]:
# One-hot encode categorical features
# One-hot encoding is used because the categories are nominal and label encoding would
# imply a ranking
cat_cols = df.select_dtypes(include='object').columns.tolist()
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Confirming the changes
print("Final dataset shape:", df_encoded.shape)
print()
print("Churn distribution:\n",df_encoded['Churn'].value_counts())

Final dataset shape: (7043, 31)

Churn distribution:
 Churn
0    5174
1    1869
Name: count, dtype: int64


## Step 3: Train/test split

In [107]:
# Train/test split
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

In [108]:
# stratify=y keeps the same 26.5% churn ratio in both training and test sets
# since the classes are not balanced 73.5% no / 26.5% yes.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("\nTrain shape:", X_train.shape, "Test shape:", X_test.shape)


Train shape: (5634, 30) Test shape: (1409, 30)


## Step 4: Decision Tree Model

In [112]:
# Model 1 = Decision Tree
# Max depth=5 limit the tree depth to avoid overfitting, if overfitting occurs
# tree will memorize the training data and generalise poorly to new data
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:,1]

print("\nDecision Tree Results")
print("Accuracy: %.3f" % accuracy_score(y_test, y_pred_dt))
print("Precision: %.3f" % precision_score(y_test, y_pred_dt))
print("Recall: %.3f" % recall_score(y_test, y_pred_dt))
print("F1 score: %.3f" % f1_score(y_test, y_pred_dt))
print("ROC-AUC:   %.3f" % roc_auc_score(y_test, y_proba_dt))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))


Decision Tree Results
Accuracy: 0.794
Precision: 0.631
Recall: 0.540
F1 score: 0.582
ROC-AUC:   0.827

Confusion Matrix:
 [[917 118]
 [172 202]]


In [110]:
# checking which features played major roles in the decision tree
importances = pd.Series(dt.feature_importances_, index=X_train.columns) \
                .sort_values(ascending=False)
print("\nTop 5 most important features (Decision Tree):")
print(importances.head(5))


Top 5 most important features (Decision Tree):
tenure                            0.422047
InternetService_Fiber optic       0.358061
TotalCharges                      0.037156
PaymentMethod_Electronic check    0.036814
MonthlyCharges                    0.025461
dtype: float64


## Step 5: SVM Model

In [113]:
# Model 2 = SVM
# SVM is distance-based model, so features must be standardised for example mean=0, std=1 otherwise large-scale features 
# e.g. (TotalCharges 0-8000) would dominate small-scale features (e.g. SeniorCitizen 0/1) purely due to units.
# Decision Trees don't need this since they split per-feature independently.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # transforming only, fit on train only, to avoid data leakage

svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)
y_proba_svm = svm.predict_proba(X_test_scaled)[:, 1]

print("\nSVM Results")
print("Accuracy: %.3f" % accuracy_score(y_test, y_pred_svm))
print("Precision: %.3f" % precision_score(y_test, y_pred_svm))
print("Recall: %.3f" % recall_score(y_test, y_pred_svm))
print("F1 score: %.3f" % f1_score(y_test, y_pred_svm))
print("ROC-AUC: %.3f" % roc_auc_score(y_test, y_proba_svm))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))


SVM Results
Accuracy: 0.793
Precision: 0.644
Recall: 0.489
F1 score: 0.556
ROC-AUC: 0.796

Confusion Matrix:
 [[934 101]
 [191 183]]
